# B2-023-generative-models-diffusion — Practice p04 — Solution

**Type:** mc · **Difficulty:** core · **Concepts:** denoising-diffusion-probabilistic-models

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Solution

**Answer: B.** The network receives the noisy row $x_t$ together with the scaled timestep $t/T$, and its regression target is the noise $\varepsilon$ that formed $x_t$ via $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon$.

Why the others fail: A feeds the clean row (unavailable at sampling time, where only $x_t$ exists) and regresses on a random quantity; C regresses on $x_0$ (see (iii)); D regresses on $x_{t-1}$, which is not what the sampler's mean formula expects; E gives the network only the noise, so it learns nothing about undoing the corruption and cannot be used by the reverse process.

**(i) Shapes for $B=256$.** `x_t` has shape `(256,2)` and `t_scaled = (t/T).unsqueeze(1)` has shape `(256,1)`; the MLP sees their concatenation, shape `(256,3)`. The output `eps_hat` has shape `(256,2)`, the same as the target `eps`.

**(ii) Why each $t\ge2$ term becomes a weighted squared error.** With $\sigma_t^2=\tilde\beta_t$, the term $\mathrm{KL}(q(x_{t-1}\mid x_t,x_0)\,\|\,p_\theta(x_{t-1}\mid x_t))$ is a KL between Gaussians of equal variance $\tilde\beta_tI$, which equals $\lVert\tilde\mu_t-\mu_\theta\rVert^2/(2\tilde\beta_t)$.
Because $\tilde\mu_t=\frac1{\sqrt{\alpha_t}}\big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\varepsilon\big)$ and $\mu_\theta$ uses the same formula with $\varepsilon_\theta$, the means differ by the schedule-only constant $\frac{\beta_t}{\sqrt{\alpha_t}\sqrt{1-\bar\alpha_t}}$ times $(\varepsilon-\varepsilon_\theta)$, so the term is $\lambda_t\lVert\varepsilon-\varepsilon_\theta(x_t,t)\rVert^2$.

**(iii) Why not option C here.** Predicting $x_0$ is a legitimate alternative parameterization (one can compute $\tilde\mu_t$ from an estimate $\hat x_0$), but it gives a differently weighted objective and a different network output. This unit's pinned training contract (`ddpm_loss`) and its sampler's mean formula $\frac1{\sqrt{\alpha_t}}(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\hat\varepsilon)$ both assume the network outputs a **noise** prediction, so a network trained on $x_0$ would be plugged into the wrong formula.

In [ ]:
import torch
from torch import nn

torch.manual_seed(20261015)
net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
B, T = 256, 50
x_t = torch.randn(B, 2, generator=torch.Generator().manual_seed(20261015))
t = torch.randint(1, T + 1, (B,), generator=torch.Generator().manual_seed(20261016))
t_scaled = (t.to(torch.float32) / T).unsqueeze(1)
net_input = torch.cat([x_t, t_scaled], dim=1)
eps_hat = net(net_input)
ANSWER = "B"
print(tuple(x_t.shape), tuple(t_scaled.shape), tuple(net_input.shape), tuple(eps_hat.shape))

### Answer check

In [ ]:
assert tuple(x_t.shape) == (256, 2) and tuple(t_scaled.shape) == (256, 1)
assert tuple(net_input.shape) == (256, 3)
assert tuple(eps_hat.shape) == (256, 2)
assert float(t_scaled.min()) > 0.0 and float(t_scaled.max()) <= 1.0
assert ANSWER == "B"